# Medical Diagnosis Expert System
Comparison of Rule-Based Expert System, Forward Chaining, Backward Chaining, and Fuzzy Logic for Clinical Decision Support.

This project implements and evaluates four knowledge-based artificial intelligence methods for clinical diagnosis using the 27 formal rules defined in rules.txt. The system evaluates the 200 patient records from medical_diagnosis_200.csv, benchmarks computational and diagnostic performance, and provides a unified interactive Gradio interface with dynamic method selection.


In [ ]:
import sys
import os
import subprocess

# install dependencies if running in colab
for pkg in ["gradio", "pandas", "numpy", "matplotlib"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import json

# candidate filenames uploaded to colab sidebar
sidebar_candidates = [
    "medical_diagnosis_200.csv",
    "medical_diagnosis_200_enriched.csv",
    "medical_diagnosis_200 (1).csv",
    "/content/medical_diagnosis_200.csv",
    "/content/medical_diagnosis_200_enriched.csv",
    "/content/medical_diagnosis_200 (1).csv"
]

dataset_path = None
for candidate in sidebar_candidates:
    if os.path.exists(candidate):
        dataset_path = candidate
        break

if dataset_path is not None:
    df = pd.read_csv(dataset_path)
    print(f"Loaded dataset from sidebar upload: {dataset_path}")
else:
    # interactive upload prompt if file not found in sidebar
    try:
        from google.colab import files
        print("CSV file not found in sidebar. Please upload medical_diagnosis_200.csv:")
        uploaded = files.upload()
        for filename in uploaded.keys():
            if filename.endswith(".csv"):
                dataset_path = filename
                df = pd.read_csv(dataset_path)
                print(f"Loaded uploaded file: {dataset_path}")
                break
    except ImportError:
        # remote fallback for non-interactive execution
        remote_url = "https://raw.githubusercontent.com/royaldinan/medical-diagnosis-expert-system/main/medical_diagnosis_200.csv"
        df = pd.read_csv(remote_url)
        print("Loaded dataset from repository fallback.")

# verify and normalize required clinical columns
required_defaults = {
    "fever_C": 37.0, "cough": 0, "fatigue": 0, "headache": 0, "nausea": 0,
    "oxygen_spo2": 98, "heart_rate": 75, "rash": "no", "bleeding": "no",
    "diarrhea": "no", "vomiting": "no", "sore_throat": "no", "body_pain": "no",
    "stomach_pain": 0, "appetite_loss": "no", "wheezing": "no", "chest_pain": "no",
    "shortness_of_breath": "no", "platelet": 250000, "joint_pain": 0,
    "night_sweats": "no", "stiff_neck": "no", "dizziness": "no", "confusion": "no",
    "sweating": "no", "blood_pressure": 120, "thirst": 0, "urination": "normal",
    "weight_loss": "no", "skin_rash": "no", "itching": "no", "eye_redness": "no",
    "watery_eyes": "no", "back_pain": 0, "stiffness": "no"
}

for col, val in required_defaults.items():
    if col not in df.columns:
        df[col] = val

if "patient_id" not in df.columns:
    df["patient_id"] = list(range(1, len(df) + 1))
if "diagnosis" not in df.columns:
    df["diagnosis"] = "Undetermined"

print(f"Dataset ready: {df.shape[0]} patients, {df.shape[1]} columns.")
df[["patient_id", "diagnosis", "fever_C", "cough", "oxygen_spo2", "heart_rate", "platelet"]].head(5)


In [ ]:
# structured clinical knowledge base representing 27 rules from rules.txt
CLINICAL_RULES = [
    # 1. common infections
    {
        "id": "R01", "disease": "Flu", "category": "Common infections",
        "conditions": [{"var": "fever", "op": ">", "val": 38.0}, {"var": "cough", "op": ">", "val": 6}],
        "contradictions": ["stiff_neck", "vomiting", "diarrhea", "bleeding", "confusion", "chest_pain", "wheezing"],
        "desc": "fever > 38 AND cough > 6"
    },
    {
        "id": "R02", "disease": "Measles", "category": "Common infections",
        "conditions": [{"var": "fever", "op": ">", "val": 39.0}, {"var": "rash", "op": "==", "val": "yes"}],
        "contradictions": ["diarrhea", "vomiting", "stiff_neck", "chest_pain"],
        "desc": "fever > 39 AND rash == 'yes'"
    },
    {
        "id": "R03", "disease": "Common Cold", "category": "Common infections",
        "conditions": [{"var": "fever", "op": ">", "val": 37.0}, {"var": "sore_throat", "op": "==", "val": "yes"}, {"var": "cough", "op": ">", "val": 4}],
        "contradictions": ["stiff_neck", "bleeding", "vomiting", "diarrhea", "shortness_of_breath", "chest_pain"],
        "desc": "fever > 37 AND sore_throat == 'yes' AND cough > 4"
    },
    {
        "id": "R04", "disease": "Viral Fever", "category": "Common infections",
        "conditions": [{"var": "fever", "op": ">", "val": 38.0}, {"var": "fatigue", "op": ">", "val": 6}, {"var": "body_pain", "op": "==", "val": "yes"}],
        "contradictions": ["stiff_neck", "bleeding", "wheezing", "chest_pain"],
        "desc": "fever > 38 AND fatigue > 6 AND body_pain == 'yes'"
    },

    # 2. gastrointestinal
    {
        "id": "R05", "disease": "Food Poisoning", "category": "Gastrointestinal",
        "conditions": [{"var": "nausea", "op": ">", "val": 6}, {"var": "vomiting", "op": "==", "val": "yes"}, {"var": "diarrhea", "op": "==", "val": "yes"}],
        "contradictions": ["stiff_neck", "wheezing", "chest_pain", "sore_throat"],
        "desc": "nausea > 6 AND vomiting == 'yes' AND diarrhea == 'yes'"
    },
    {
        "id": "R06", "disease": "Gastroenteritis", "category": "Gastrointestinal",
        "conditions": [{"var": "stomach_pain", "op": ">", "val": 7}, {"var": "vomiting", "op": "==", "val": "yes"}, {"var": "fever", "op": ">", "val": 37.5}],
        "contradictions": ["stiff_neck", "wheezing", "chest_pain", "sore_throat"],
        "desc": "stomach_pain > 7 AND vomiting == 'yes' AND fever > 37.5"
    },
    {
        "id": "R07", "disease": "Indigestion", "category": "Gastrointestinal",
        "conditions": [{"var": "nausea", "op": ">", "val": 5}, {"var": "appetite_loss", "op": "==", "val": "yes"}],
        "contradictions": ["bleeding", "stiff_neck", "wheezing"],
        "desc": "nausea > 5 AND appetite_loss == 'yes'"
    },

    # 3. respiratory
    {
        "id": "R08", "disease": "Pneumonia", "category": "Respiratory",
        "conditions": [{"var": "oxygen_spo2", "op": "<", "val": 92}, {"var": "cough", "op": ">", "val": 7}, {"var": "fever", "op": ">", "val": 38.0}],
        "contradictions": ["stiff_neck", "bleeding", "diarrhea"],
        "desc": "oxygen_spo2 < 92 AND cough > 7 AND fever > 38"
    },
    {
        "id": "R09", "disease": "Bronchitis", "category": "Respiratory",
        "conditions": [{"var": "cough", "op": ">", "val": 8}, {"var": "wheezing", "op": "==", "val": "yes"}, {"var": "chest_pain", "op": "==", "val": "yes"}],
        "contradictions": ["stiff_neck", "bleeding", "diarrhea"],
        "desc": "cough > 8 AND wheezing == 'yes' AND chest_pain == 'yes'"
    },
    {
        "id": "R10", "disease": "Asthma", "category": "Respiratory",
        "conditions": [{"var": "cough", "op": ">", "val": 6}, {"var": "shortness_of_breath", "op": "==", "val": "yes"}, {"var": "fatigue", "op": ">", "val": 6}],
        "contradictions": ["bleeding", "stiff_neck", "vomiting", "fever"],
        "desc": "cough > 6 AND shortness_of_breath == 'yes' AND fatigue > 6"
    },

    # 4. vector-borne
    {
        "id": "R11", "disease": "Dengue", "category": "Vector-borne",
        "conditions": [{"var": "fever", "op": ">", "val": 38.0}, {"var": "headache", "op": ">", "val": 6}, {"var": "rash", "op": "==", "val": "yes"}],
        "contradictions": ["stiff_neck", "wheezing", "shortness_of_breath", "bleeding"],
        "desc": "fever > 38 AND headache > 6 AND rash == 'yes'"
    },
    {
        "id": "R12", "disease": "Dengue Hemorrhagic Fever", "category": "Vector-borne",
        "conditions": [{"var": "fever", "op": ">", "val": 39.0}, {"var": "bleeding", "op": "==", "val": "yes"}, {"var": "platelet", "op": "<", "val": 100000}],
        "contradictions": ["stiff_neck", "wheezing"],
        "desc": "fever > 39 AND bleeding == 'yes' AND platelet < 100000"
    },
    {
        "id": "R13", "disease": "Chikungunya", "category": "Vector-borne",
        "conditions": [{"var": "fever", "op": ">", "val": 37.5}, {"var": "joint_pain", "op": ">", "val": 7}],
        "contradictions": ["stiff_neck", "bleeding", "shortness_of_breath"],
        "desc": "fever > 37.5 AND joint_pain > 7"
    },
    {
        "id": "R14", "disease": "Tuberculosis", "category": "Vector-borne",
        "conditions": [{"var": "fever", "op": ">", "val": 38.0}, {"var": "night_sweats", "op": "==", "val": "yes"}, {"var": "cough", "op": ">", "val": 5}],
        "contradictions": ["stiff_neck", "bleeding"],
        "desc": "fever > 38 AND night_sweats == 'yes' AND cough > 5"
    },

    # 5. neurological
    {
        "id": "R15", "disease": "Meningitis", "category": "Neurological",
        "conditions": [{"var": "headache", "op": ">", "val": 8}, {"var": "fever", "op": ">", "val": 38.0}, {"var": "stiff_neck", "op": "==", "val": "yes"}],
        "contradictions": ["wheezing", "itching"],
        "desc": "headache > 8 AND fever > 38 AND stiff_neck == 'yes'"
    },
    {
        "id": "R16", "disease": "Migraine", "category": "Neurological",
        "conditions": [{"var": "headache", "op": ">", "val": 7}, {"var": "dizziness", "op": "==", "val": "yes"}, {"var": "fatigue", "op": ">", "val": 6}],
        "contradictions": ["fever", "stiff_neck", "shortness_of_breath"],
        "desc": "headache > 7 AND dizziness == 'yes' AND fatigue > 6"
    },
    {
        "id": "R17", "disease": "Chronic Fatigue Syndrome", "category": "Neurological",
        "conditions": [{"var": "fatigue", "op": ">", "val": 8}, {"var": "confusion", "op": "==", "val": "yes"}],
        "contradictions": ["fever", "bleeding", "stiff_neck"],
        "desc": "fatigue > 8 AND confusion == 'yes'"
    },

    # 6. cardiovascular
    {
        "id": "R18", "disease": "Cardiac Stress", "category": "Cardiovascular",
        "conditions": [{"var": "heart_rate", "op": ">", "val": 120}, {"var": "fatigue", "op": ">", "val": 8}],
        "contradictions": ["fever", "rash"],
        "desc": "heart_rate > 120 AND fatigue > 8"
    },
    {
        "id": "R19", "disease": "Heart Attack", "category": "Cardiovascular",
        "conditions": [{"var": "chest_pain", "op": "==", "val": "yes"}, {"var": "shortness_of_breath", "op": "==", "val": "yes"}, {"var": "sweating", "op": "==", "val": "yes"}],
        "contradictions": ["fever", "rash", "itching"],
        "desc": "chest_pain == 'yes' AND shortness_of_breath == 'yes' AND sweating == 'yes'"
    },
    {
        "id": "R20", "disease": "Hypertension", "category": "Cardiovascular",
        "conditions": [{"var": "blood_pressure", "op": ">", "val": 140}, {"var": "headache", "op": ">", "val": 6}],
        "contradictions": ["fever", "rash"],
        "desc": "blood_pressure > 140 AND headache > 6"
    },

    # 7. metabolic
    {
        "id": "R21", "disease": "Diabetes", "category": "Metabolic",
        "conditions": [{"var": "thirst", "op": ">", "val": 7}, {"var": "urination", "op": "==", "val": "frequent"}, {"var": "fatigue", "op": ">", "val": 5}],
        "contradictions": ["fever", "stiff_neck"],
        "desc": "thirst > 7 AND urination == 'frequent' AND fatigue > 5"
    },
    {
        "id": "R22", "disease": "Thyroid Disorder", "category": "Metabolic",
        "conditions": [{"var": "weight_loss", "op": "==", "val": "yes"}, {"var": "appetite_loss", "op": "==", "val": "yes"}, {"var": "fatigue", "op": ">", "val": 7}],
        "contradictions": ["fever", "stiff_neck"],
        "desc": "weight_loss == 'yes' AND appetite_loss == 'yes' AND fatigue > 7"
    },

    # 8. others
    {
        "id": "R23", "disease": "Allergic Reaction", "category": "Others",
        "conditions": [{"var": "skin_rash", "op": "==", "val": "yes"}, {"var": "itching", "op": "==", "val": "yes"}],
        "contradictions": ["fever", "stiff_neck", "bleeding"],
        "desc": "skin_rash == 'yes' AND itching == 'yes'"
    },
    {
        "id": "R24", "disease": "Conjunctivitis", "category": "Others",
        "conditions": [{"var": "eye_redness", "op": "==", "val": "yes"}, {"var": "watery_eyes", "op": "==", "val": "yes"}],
        "contradictions": ["fever", "chest_pain"],
        "desc": "eye_redness == 'yes' AND watery_eyes == 'yes'"
    },
    {
        "id": "R25", "disease": "Muscle Strain", "category": "Others",
        "conditions": [{"var": "back_pain", "op": ">", "val": 7}, {"var": "fatigue", "op": ">", "val": 6}],
        "contradictions": ["fever", "bleeding", "stiff_neck"],
        "desc": "back_pain > 7 AND fatigue > 6"
    },
    {
        "id": "R26", "disease": "Arthritis", "category": "Others",
        "conditions": [{"var": "joint_pain", "op": ">", "val": 6}, {"var": "stiffness", "op": "==", "val": "yes"}],
        "contradictions": ["fever", "chest_pain", "shortness_of_breath"],
        "desc": "joint_pain > 6 AND stiffness == 'yes'"
    },
    {
        "id": "R27", "disease": "Healthy", "category": "Others",
        "conditions": [{"var": "fever", "op": "<", "val": 37.5}, {"var": "all_symptoms", "op": "==", "val": "low"}],
        "contradictions": [],
        "desc": "fever < 37.5 AND all symptoms == 'low'"
    }
]

def normalize_patient(raw):
    p = {}
    p["fever"] = float(raw.get("fever_C", raw.get("fever", 37.0)))
    p["cough"] = int(float(raw.get("cough", 0)))
    p["fatigue"] = int(float(raw.get("fatigue", 0)))
    p["headache"] = int(float(raw.get("headache", 0)))
    p["nausea"] = int(float(raw.get("nausea", 0)))
    p["oxygen_spo2"] = int(float(raw.get("oxygen_spo2", 98)))
    p["heart_rate"] = int(float(raw.get("heart_rate", 75)))
    p["stomach_pain"] = int(float(raw.get("stomach_pain", 0)))
    p["platelet"] = int(float(raw.get("platelet", 250000)))
    p["joint_pain"] = int(float(raw.get("joint_pain", 0)))
    p["back_pain"] = int(float(raw.get("back_pain", 0)))
    p["thirst"] = int(float(raw.get("thirst", 0)))
    p["blood_pressure"] = int(float(raw.get("blood_pressure", 120)))

    str_keys = [
        "rash", "bleeding", "diarrhea", "vomiting", "sore_throat", "body_pain",
        "appetite_loss", "wheezing", "chest_pain", "shortness_of_breath",
        "night_sweats", "stiff_neck", "dizziness", "confusion", "sweating",
        "urination", "weight_loss", "skin_rash", "itching", "eye_redness",
        "watery_eyes", "stiffness"
    ]
    for k in str_keys:
        v = str(raw.get(k, "no")).strip().lower()
        p[k] = v if v else "no"
    if p["rash"] == "yes" and p["skin_rash"] == "no":
        p["skin_rash"] = "yes"
    return p

def eval_single_condition(c, p):
    var = c["var"]
    op = c["op"]
    val = c["val"]
    if var == "all_symptoms" and val == "low":
        symptom_scalars = [
            p["cough"], p["fatigue"], p["headache"], p["nausea"],
            p["stomach_pain"], p["joint_pain"], p["back_pain"], p["thirst"]
        ]
        red_flags = [
            p["rash"], p["vomiting"], p["diarrhea"], p["chest_pain"],
            p["stiff_neck"], p["bleeding"], p["wheezing"], p["shortness_of_breath"]
        ]
        return max(symptom_scalars) <= 2 and all(v == "no" for v in red_flags)
    
    curr = p.get(var)
    if curr is None:
        return False
    if op == ">":
        return float(curr) > float(val)
    elif op == "<":
        return float(curr) < float(val)
    elif op == "==":
        return str(curr).strip().lower() == str(val).strip().lower()
    return False

def check_contradictions(rule, p):
    contradictions = rule.get("contradictions", [])
    active_contra = []
    for c in contradictions:
        if c == "fever":
            if p.get("fever", 37.0) > 37.8:
                active_contra.append(f"fever({p['fever']}C)")
        elif c in ["fatigue", "headache", "nausea", "stomach_pain", "joint_pain", "back_pain"]:
            if p.get(c, 0) > 6:
                active_contra.append(f"{c}({p[c]})")
        elif c == "bleeding":
            if p.get("bleeding", "no") == "yes":
                active_contra.append("bleeding")
        elif c == "stiff_neck":
            if p.get("stiff_neck", "no") == "yes":
                active_contra.append("stiff_neck")
        elif c == "vomiting":
            if p.get("vomiting", "no") == "yes":
                active_contra.append("vomiting")
        elif c == "diarrhea":
            if p.get("diarrhea", "no") == "yes":
                active_contra.append("diarrhea")
        elif c == "chest_pain":
            if p.get("chest_pain", "no") == "yes":
                active_contra.append("chest_pain")
        elif c == "wheezing":
            if p.get("wheezing", "no") == "yes":
                active_contra.append("wheezing")
        elif c == "shortness_of_breath":
            if p.get("shortness_of_breath", "no") == "yes":
                active_contra.append("shortness_of_breath")
        elif c == "confusion":
            if p.get("confusion", "no") == "yes":
                active_contra.append("confusion")
        else:
            if str(p.get(c, "no")).strip().lower() == "yes":
                active_contra.append(c)
    return active_contra


In [ ]:
# method 1: rule-based expert system
class RuleBasedExpertSystem:
    def __init__(self, rules):
        self.rules = rules

    def diagnose(self, raw_patient):
        p = normalize_patient(raw_patient)
        fired = []

        for r in self.rules:
            all_met = all(eval_single_condition(c, p) for c in r["conditions"])
            if all_met:
                contra = check_contradictions(r, p)
                cond_count = len(r["conditions"])
                penalty = min(0.60, len(contra) * 0.20)
                cf = max(0.15, (0.82 + 0.05 * min(3, cond_count)) * (1.0 - penalty))
                fired.append({
                    "rule": r,
                    "cond_count": cond_count,
                    "contra": contra,
                    "cf": round(cf, 2)
                })

        if not fired:
            return {
                "method": "Rule-Based Expert System",
                "diagnosis": "Undetermined / Inconclusive",
                "confidence": 0.0,
                "primary_rule": None,
                "fired_rules": [],
                "reasoning": "No clinical production rule fired for the observed symptom profile."
            }

        # resolve conflicts via precondition specificity and contradiction penalties
        fired.sort(key=lambda x: (x["cond_count"], -len(x["contra"]), x["cf"]), reverse=True)
        winner = fired[0]
        w_rule = winner["rule"]

        notes = []
        if winner["contra"]:
            notes.append("Incongruent clinical signs flagged: " + ", ".join(winner["contra"]))

        return {
            "method": "Rule-Based Expert System",
            "diagnosis": w_rule["disease"],
            "confidence": winner["cf"],
            "primary_rule": w_rule["id"],
            "fired_rules": [f"{item['rule']['id']}: {item['rule']['disease']} (CF: {item['cf']}, Specificity: {item['cond_count']})" for item in fired],
            "reasoning": f"Fired {w_rule['id']} ({w_rule['desc']}) with specificity {winner['cond_count']} and Certainty Factor {winner['cf']}." + (" " + " ".join(notes) if notes else "")
        }

rbs_engine = RuleBasedExpertSystem(CLINICAL_RULES)


In [ ]:
# method 2: forward chaining inference engine
class ForwardChainingEngine:
    def __init__(self, rules):
        self.rules = rules

    def infer(self, raw_patient):
        p = normalize_patient(raw_patient)
        wm = dict(p)
        trace = []
        cycle = 1

        # cycle 1: abstract raw continuous vitals and signs into categorical clinical states
        c1_rules = [
            ("hyperthermia", lambda m: m["fever"] > 38.0, "fever > 38.0"),
            ("mild_pyrexia", lambda m: 37.0 < m["fever"] <= 38.0, "37.0 < fever <= 38.0"),
            ("high_grade_fever", lambda m: m["fever"] > 39.0, "fever > 39.0"),
            ("normal_temp", lambda m: m["fever"] < 37.5, "fever < 37.5"),
            ("severe_cough", lambda m: m["cough"] > 6, "cough > 6"),
            ("moderate_cough", lambda m: 4 < m["cough"] <= 6, "4 < cough <= 6"),
            ("hypoxemia", lambda m: m["oxygen_spo2"] < 92, "oxygen_spo2 < 92"),
            ("severe_fatigue", lambda m: m["fatigue"] > 6, "fatigue > 6"),
            ("critical_fatigue", lambda m: m["fatigue"] > 8, "fatigue > 8"),
            ("severe_headache", lambda m: m["headache"] > 6, "headache > 6"),
            ("critical_headache", lambda m: m["headache"] > 8, "headache > 8"),
            ("severe_nausea", lambda m: m["nausea"] > 6, "nausea > 6"),
            ("moderate_nausea", lambda m: 5 < m["nausea"] <= 6, "5 < nausea <= 6"),
            ("severe_abdominal_pain", lambda m: m["stomach_pain"] > 7, "stomach_pain > 7"),
            ("severe_arthralgia", lambda m: m["joint_pain"] > 7, "joint_pain > 7"),
            ("moderate_arthralgia", lambda m: m["joint_pain"] > 6, "joint_pain > 6"),
            ("severe_lumbar_pain", lambda m: m["back_pain"] > 7, "back_pain > 7"),
            ("tachycardia", lambda m: m["heart_rate"] > 120, "heart_rate > 120"),
            ("hypertensive_bp", lambda m: m["blood_pressure"] > 140, "blood_pressure > 140"),
            ("thrombocytopenia", lambda m: m["platelet"] < 100000, "platelet < 100000"),
            ("polydipsia", lambda m: m["thirst"] > 7, "thirst > 7")
        ]

        c1_asserted = []
        for fact, cond_fn, reason in c1_rules:
            if cond_fn(wm):
                wm[fact] = True
                c1_asserted.append(f"{fact} [{reason}]")

        if c1_asserted:
            trace.append(f"Cycle {cycle} (Clinical Abstraction): Asserted {', '.join(c1_asserted)}")
            cycle += 1

        # cycle 2: combine base states into intermediate clinical syndromes
        c2_rules = [
            ("acute_respiratory_syndrome", lambda m: m.get("hyperthermia") and (m.get("severe_cough") or m.get("moderate_cough")), "hyperthermia AND cough"),
            ("lower_respiratory_distress", lambda m: m.get("acute_respiratory_syndrome") and m.get("hypoxemia"), "acute_respiratory_syndrome AND hypoxemia"),
            ("meningeal_syndrome", lambda m: m.get("critical_headache") and m.get("hyperthermia") and m.get("stiff_neck") == "yes", "critical_headache AND hyperthermia AND stiff_neck"),
            ("bronchial_syndrome", lambda m: (m.get("severe_cough") or m["cough"] > 8) and m.get("wheezing") == "yes" and m.get("chest_pain") == "yes", "severe_cough AND wheezing AND chest_pain"),
            ("food_poisoning_syndrome", lambda m: m.get("severe_nausea") and m.get("vomiting") == "yes" and m.get("diarrhea") == "yes", "severe_nausea AND vomiting AND diarrhea"),
            ("gastroenteritis_syndrome", lambda m: m.get("severe_abdominal_pain") and m.get("vomiting") == "yes" and m["fever"] > 37.5, "severe_abdominal_pain AND vomiting AND fever > 37.5"),
            ("dengue_syndrome", lambda m: m.get("hyperthermia") and m.get("severe_headache") and m.get("rash") == "yes", "hyperthermia AND severe_headache AND rash"),
            ("dengue_hemorrhagic_syndrome", lambda m: m.get("high_grade_fever") and m.get("thrombocytopenia") and m.get("bleeding") == "yes", "high_grade_fever AND thrombocytopenia AND bleeding"),
            ("cardiac_crisis_syndrome", lambda m: m.get("chest_pain") == "yes" and m.get("shortness_of_breath") == "yes" and m.get("sweating") == "yes", "chest_pain AND shortness_of_breath AND sweating"),
            ("common_cold_syndrome", lambda m: (m.get("mild_pyrexia") or m.get("hyperthermia") or m["fever"] > 37.0) and m.get("sore_throat") == "yes" and (m.get("moderate_cough") or m.get("severe_cough")), "fever > 37 AND sore_throat AND cough > 4")
        ]

        c2_asserted = []
        for syn, cond_fn, reason in c2_rules:
            if cond_fn(wm):
                wm[syn] = True
                c2_asserted.append(f"{syn} [{reason}]")

        if c2_asserted:
            trace.append(f"Cycle {cycle} (Syndrome Deduction): Asserted {', '.join(c2_asserted)}")
            cycle += 1

        # cycle 3: deduce terminal disease STRICTLY FROM working memory assertions
        terminal_candidates = []

        if wm.get("lower_respiratory_distress"):
            terminal_candidates.append({"disease": "Pneumonia", "rule_id": "R08", "spec": 4, "contra": check_contradictions(self.rules[7], p)})
        if wm.get("meningeal_syndrome"):
            terminal_candidates.append({"disease": "Meningitis", "rule_id": "R15", "spec": 4, "contra": check_contradictions(self.rules[14], p)})
        if wm.get("dengue_hemorrhagic_syndrome"):
            terminal_candidates.append({"disease": "Dengue Hemorrhagic Fever", "rule_id": "R12", "spec": 4, "contra": check_contradictions(self.rules[11], p)})
        if wm.get("bronchial_syndrome"):
            terminal_candidates.append({"disease": "Bronchitis", "rule_id": "R09", "spec": 3, "contra": check_contradictions(self.rules[8], p)})
        if wm.get("food_poisoning_syndrome"):
            terminal_candidates.append({"disease": "Food Poisoning", "rule_id": "R05", "spec": 3, "contra": check_contradictions(self.rules[4], p)})
        if wm.get("gastroenteritis_syndrome"):
            terminal_candidates.append({"disease": "Gastroenteritis", "rule_id": "R06", "spec": 3, "contra": check_contradictions(self.rules[5], p)})
        if wm.get("cardiac_crisis_syndrome"):
            terminal_candidates.append({"disease": "Heart Attack", "rule_id": "R19", "spec": 3, "contra": check_contradictions(self.rules[18], p)})
        if wm.get("dengue_syndrome") and not wm.get("thrombocytopenia"):
            terminal_candidates.append({"disease": "Dengue", "rule_id": "R11", "spec": 3, "contra": check_contradictions(self.rules[10], p)})
        if wm.get("common_cold_syndrome") and not wm.get("meningeal_syndrome"):
            terminal_candidates.append({"disease": "Common Cold", "rule_id": "R03", "spec": 3, "contra": check_contradictions(self.rules[2], p)})
        if wm.get("hyperthermia") and wm.get("severe_fatigue") and wm.get("body_pain") == "yes":
            terminal_candidates.append({"disease": "Viral Fever", "rule_id": "R04", "spec": 3, "contra": check_contradictions(self.rules[3], p)})
        if wm.get("acute_respiratory_syndrome") and not wm.get("hypoxemia") and not wm.get("meningeal_syndrome"):
            terminal_candidates.append({"disease": "Flu", "rule_id": "R01", "spec": 2, "contra": check_contradictions(self.rules[0], p)})
        if wm.get("high_grade_fever") and wm.get("rash") == "yes":
            terminal_candidates.append({"disease": "Measles", "rule_id": "R02", "spec": 2, "contra": check_contradictions(self.rules[1], p)})
        if (wm.get("severe_nausea") or wm.get("moderate_nausea")) and wm.get("appetite_loss") == "yes":
            terminal_candidates.append({"disease": "Indigestion", "rule_id": "R07", "spec": 2, "contra": check_contradictions(self.rules[6], p)})
        if (wm.get("severe_cough") or p["cough"] > 6) and wm.get("shortness_of_breath") == "yes" and wm.get("severe_fatigue"):
            terminal_candidates.append({"disease": "Asthma", "rule_id": "R10", "spec": 3, "contra": check_contradictions(self.rules[9], p)})
        if (wm.get("hyperthermia") or p["fever"] > 37.5) and wm.get("severe_arthralgia"):
            terminal_candidates.append({"disease": "Chikungunya", "rule_id": "R13", "spec": 2, "contra": check_contradictions(self.rules[12], p)})
        if wm.get("hyperthermia") and wm.get("night_sweats") == "yes" and (wm.get("moderate_cough") or wm.get("severe_cough") or p["cough"] > 5):
            terminal_candidates.append({"disease": "Tuberculosis", "rule_id": "R14", "spec": 3, "contra": check_contradictions(self.rules[13], p)})
        if wm.get("severe_headache") and wm.get("dizziness") == "yes" and wm.get("severe_fatigue") and not wm.get("hyperthermia"):
            terminal_candidates.append({"disease": "Migraine", "rule_id": "R16", "spec": 3, "contra": check_contradictions(self.rules[15], p)})
        if wm.get("critical_fatigue") and wm.get("confusion") == "yes":
            terminal_candidates.append({"disease": "Chronic Fatigue Syndrome", "rule_id": "R17", "spec": 2, "contra": check_contradictions(self.rules[16], p)})
        if wm.get("tachycardia") and wm.get("critical_fatigue"):
            terminal_candidates.append({"disease": "Cardiac Stress", "rule_id": "R18", "spec": 2, "contra": check_contradictions(self.rules[17], p)})
        if wm.get("hypertensive_bp") and wm.get("severe_headache"):
            terminal_candidates.append({"disease": "Hypertension", "rule_id": "R20", "spec": 2, "contra": check_contradictions(self.rules[19], p)})
        if wm.get("polydipsia") and wm.get("urination") == "frequent" and (wm.get("severe_fatigue") or p["fatigue"] > 5):
            terminal_candidates.append({"disease": "Diabetes", "rule_id": "R21", "spec": 3, "contra": check_contradictions(self.rules[20], p)})
        if wm.get("weight_loss") == "yes" and wm.get("appetite_loss") == "yes" and (wm.get("severe_fatigue") or p["fatigue"] > 7):
            terminal_candidates.append({"disease": "Thyroid Disorder", "rule_id": "R22", "spec": 3, "contra": check_contradictions(self.rules[21], p)})
        if wm.get("skin_rash") == "yes" and wm.get("itching") == "yes":
            terminal_candidates.append({"disease": "Allergic Reaction", "rule_id": "R23", "spec": 2, "contra": check_contradictions(self.rules[22], p)})
        if wm.get("eye_redness") == "yes" and wm.get("watery_eyes") == "yes":
            terminal_candidates.append({"disease": "Conjunctivitis", "rule_id": "R24", "spec": 2, "contra": check_contradictions(self.rules[23], p)})
        if wm.get("severe_lumbar_pain") and wm.get("severe_fatigue"):
            terminal_candidates.append({"disease": "Muscle Strain", "rule_id": "R25", "spec": 2, "contra": check_contradictions(self.rules[24], p)})
        if wm.get("moderate_arthralgia") and wm.get("stiffness") == "yes":
            terminal_candidates.append({"disease": "Arthritis", "rule_id": "R26", "spec": 2, "contra": check_contradictions(self.rules[25], p)})
        if wm.get("normal_temp") and eval_single_condition({"var": "all_symptoms", "op": "==", "val": "low"}, p):
            terminal_candidates.append({"disease": "Healthy", "rule_id": "R27", "spec": 2, "contra": []})

        if not terminal_candidates:
            trace.append(f"Cycle {cycle} (Terminal Goal): Working memory saturated without meeting terminal disease criteria.")
            return {
                "method": "Forward Chaining",
                "diagnosis": "Undetermined / Inconclusive",
                "confidence": 0.0,
                "cycles": cycle,
                "trace": trace,
                "reasoning": "Inference completed without matching terminal disease criteria in working memory."
            }

        # resolution based on specificity and least contradictions
        for cand in terminal_candidates:
            penalty = min(0.60, len(cand["contra"]) * 0.20)
            cand["cf"] = round(max(0.15, (0.85 + 0.04 * min(3, cand["spec"])) * (1.0 - penalty)), 2)

        terminal_candidates.sort(key=lambda x: (x["spec"], -len(x["contra"]), x["cf"]), reverse=True)
        winner = terminal_candidates[0]

        trace.append(f"Cycle {cycle} (Terminal Goal): Deduced {winner['disease']} from working memory (Specificity: {winner['spec']}, CF: {winner['cf']}).")

        notes = []
        if winner["contra"]:
            notes.append("Incongruent clinical signs: " + ", ".join(winner["contra"]))

        return {
            "method": "Forward Chaining",
            "diagnosis": winner["disease"],
            "confidence": winner["cf"],
            "cycles": cycle,
            "primary_rule": winner["rule_id"],
            "trace": trace,
            "reasoning": f"Forward inference chained through {cycle} cycles to confirm {winner['disease']} (Certainty Factor {winner['cf']})." + (" " + " ".join(notes) if notes else "")
        }

fc_engine = ForwardChainingEngine(CLINICAL_RULES)


In [ ]:
# method 3: backward chaining inference engine
class BackwardChainingEngine:
    def __init__(self, rules):
        self.rules = rules

    def evaluate_hypothesis(self, disease, raw_patient):
        p = normalize_patient(raw_patient)
        matched_rules = [r for r in self.rules if r["disease"].lower() == disease.lower()]
        if not matched_rules:
            return {
                "goal": disease,
                "proven": False,
                "trace": [f"No rule exists in knowledge base concluding '{disease}'."],
                "missing_conditions": [],
                "spec": 0,
                "cf": 0.0,
                "contra": []
            }

        rule = matched_rules[0]
        subgoals = []
        missing = []
        all_passed = True

        for c in rule["conditions"]:
            passed = eval_single_condition(c, p)
            var = c["var"]
            val = c["val"]
            op = c["op"]
            curr = p.get(var, "n/a")
            subgoals.append(f"Verify sub-goal [{var} {op} {val}]: Observed '{curr}' -> {'PASSED' if passed else 'FAILED'}")
            if not passed:
                all_passed = False
                missing.append(f"{var} {op} {val} (got {curr})")

        contra = check_contradictions(rule, p)
        penalty = min(0.60, len(contra) * 0.20)
        spec = len(rule["conditions"])
        cf = round(max(0.15, (0.85 + 0.04 * min(3, spec)) * (1.0 - penalty)), 2) if all_passed else 0.0

        if contra and all_passed:
            subgoals.append("Differential notice: incongruent clinical signs observed: " + ", ".join(contra))

        return {
            "goal": rule["disease"],
            "proven": all_passed,
            "rule_id": rule["id"],
            "spec": spec,
            "cf": cf,
            "trace": subgoals,
            "missing_conditions": missing,
            "contra": contra
        }

    def infer(self, raw_patient, target_hypothesis=None):
        if target_hypothesis and target_hypothesis != "Auto (Evaluate All Candidates)":
            eval_res = self.evaluate_hypothesis(target_hypothesis, raw_patient)
            return {
                "method": "Backward Chaining",
                "diagnosis": eval_res["goal"] if eval_res["proven"] else "Undetermined / Inconclusive",
                "goal_tested": eval_res["goal"],
                "proven": eval_res["proven"],
                "confidence": eval_res["cf"],
                "trace": eval_res["trace"],
                "missing": eval_res["missing_conditions"],
                "reasoning": f"Hypothesis '{target_hypothesis}' was {'PROVEN with Certainty Factor ' + str(eval_res['cf']) if eval_res['proven'] else 'REFUTED due to unmet sub-goals'}."
            }

        # auto mode: evaluate all hypotheses and resolve winner by specificity and fewest contradictions
        evaluations = []
        proven_candidates = []

        for r in self.rules:
            res = self.evaluate_hypothesis(r["disease"], raw_patient)
            evaluations.append(res)
            if res["proven"]:
                proven_candidates.append(res)

        if proven_candidates:
            proven_candidates.sort(key=lambda x: (x["spec"], -len(x["contra"]), x["cf"]), reverse=True)
            winner = proven_candidates[0]
            return {
                "method": "Backward Chaining",
                "diagnosis": winner["goal"],
                "goal_tested": winner["goal"],
                "proven": True,
                "confidence": winner["cf"],
                "trace": winner["trace"],
                "missing": [],
                "reasoning": f"Goal hypothesis '{winner['goal']}' proven across {winner['spec']} clinical sub-goals (Certainty Factor {winner['cf']})."
            }
        else:
            evaluations.sort(key=lambda x: len(x["missing_conditions"]))
            closest = evaluations[0]
            return {
                "method": "Backward Chaining",
                "diagnosis": "Undetermined / Inconclusive",
                "goal_tested": closest["goal"],
                "proven": False,
                "confidence": 0.0,
                "trace": closest["trace"],
                "missing": closest["missing_conditions"],
                "reasoning": f"Closest candidate '{closest['goal']}' refuted due to {len(closest['missing_conditions'])} unsatisfied conditions: " + ", ".join(closest["missing_conditions"]) + "."
            }

bc_engine = BackwardChainingEngine(CLINICAL_RULES)


In [ ]:
# method 4: fuzzy logic mamdani inference engine
class FuzzyLogicEngine:
    def __init__(self, rules):
        self.rules = rules

    def _ramp(self, x, low, high):
        if x <= low:
            return 0.0
        if x >= high:
            return 1.0
        return (x - low) / (high - low)

    def _inv_ramp(self, x, low, high):
        if x <= low:
            return 1.0
        if x >= high:
            return 0.0
        return (high - x) / (high - low)

    def _tri(self, x, a, b, c):
        if x <= a or x >= c:
            return 0.0
        if x == b:
            return 1.0
        if a < x < b:
            return (x - a) / (b - a)
        return (c - x) / (c - b)

    def infer(self, raw_patient):
        p = normalize_patient(raw_patient)

        fever = p["fever"]
        cough = p["cough"]
        fatigue = p["fatigue"]
        headache = p["headache"]
        nausea = p["nausea"]
        spo2 = p["oxygen_spo2"]
        hr = p["heart_rate"]
        bp = p["blood_pressure"]
        thirst = p["thirst"]
        joint = p["joint_pain"]
        back = p["back_pain"]
        stomach = p["stomach_pain"]
        platelet = p["platelet"]

        # graded continuous membership functions for vital signs and severity scales
        mf = {
            "fever_normal": self._inv_ramp(fever, 37.0, 37.5),
            "fever_mild": self._tri(fever, 37.0, 37.5, 38.2),
            "fever_high": self._ramp(fever, 37.8, 39.2),
            "fever_extreme": self._ramp(fever, 38.8, 40.0),

            "cough_mild": self._tri(cough, 1, 3, 5),
            "cough_mod": self._tri(cough, 3, 5, 7),
            "cough_severe": self._ramp(cough, 5.5, 9.5),

            "spo2_hypoxic": self._inv_ramp(spo2, 88, 93),
            "spo2_borderline": self._tri(spo2, 91, 94, 97),
            "spo2_normal": self._ramp(spo2, 94, 98),

            "fatigue_high": self._ramp(fatigue, 5.0, 8.5),
            "fatigue_extreme": self._ramp(fatigue, 7.5, 9.5),

            "headache_high": self._ramp(headache, 5.5, 8.5),
            "headache_severe": self._ramp(headache, 7.5, 9.5),

            "nausea_high": self._ramp(nausea, 5.0, 8.0),
            "stomach_high": self._ramp(stomach, 6.0, 9.0),

            "hr_tachycardia": self._ramp(hr, 100, 135),
            "bp_hypertension": self._ramp(bp, 130, 160),
            "platelet_low": self._inv_ramp(platelet, 70000, 110000),

            "thirst_high": self._ramp(thirst, 6.0, 9.0),
            "joint_high": self._ramp(joint, 5.5, 8.5),
            "back_high": self._ramp(back, 6.0, 9.0),
        }

        # scaled binary signs
        s = {k: 0.95 if p[k] == "yes" else 0.0 for k in [
            "rash", "bleeding", "diarrhea", "vomiting", "sore_throat", "body_pain",
            "appetite_loss", "wheezing", "chest_pain", "shortness_of_breath",
            "night_sweats", "stiff_neck", "dizziness", "confusion", "sweating",
            "weight_loss", "skin_rash", "itching", "eye_redness", "watery_eyes", "stiffness"
        ]}
        s["urination_freq"] = 0.95 if p["urination"] == "frequent" else 0.0

        # evaluate fuzzy production rules using t-norm min conjunction
        raw_scores = {}
        raw_scores["Flu"] = min(mf["fever_high"], mf["cough_severe"])
        raw_scores["Measles"] = min(mf["fever_extreme"], s["rash"])
        raw_scores["Common Cold"] = min(max(mf["fever_mild"], mf["fever_high"]), s["sore_throat"], max(mf["cough_mod"], mf["cough_severe"]))
        raw_scores["Viral Fever"] = min(mf["fever_high"], mf["fatigue_high"], s["body_pain"])

        raw_scores["Food Poisoning"] = min(mf["nausea_high"], s["vomiting"], s["diarrhea"])
        raw_scores["Gastroenteritis"] = min(mf["stomach_high"], s["vomiting"], max(mf["fever_mild"], mf["fever_high"]))
        raw_scores["Indigestion"] = min(mf["nausea_high"], s["appetite_loss"])

        raw_scores["Pneumonia"] = min(mf["spo2_hypoxic"], mf["cough_severe"], mf["fever_high"])
        raw_scores["Bronchitis"] = min(mf["cough_severe"], s["wheezing"], s["chest_pain"])
        raw_scores["Asthma"] = min(mf["cough_severe"], s["shortness_of_breath"], mf["fatigue_high"])

        raw_scores["Dengue"] = min(mf["fever_high"], mf["headache_high"], s["rash"])
        raw_scores["Dengue Hemorrhagic Fever"] = min(mf["fever_extreme"], s["bleeding"], mf["platelet_low"])
        raw_scores["Chikungunya"] = min(max(mf["fever_mild"], mf["fever_high"]), mf["joint_high"])
        raw_scores["Tuberculosis"] = min(mf["fever_high"], s["night_sweats"], max(mf["cough_mod"], mf["cough_severe"]))

        raw_scores["Meningitis"] = min(mf["headache_severe"], mf["fever_high"], s["stiff_neck"])
        raw_scores["Migraine"] = min(mf["headache_high"], s["dizziness"], mf["fatigue_high"])
        raw_scores["Chronic Fatigue Syndrome"] = min(mf["fatigue_extreme"], s["confusion"])

        raw_scores["Cardiac Stress"] = min(mf["hr_tachycardia"], mf["fatigue_extreme"])
        raw_scores["Heart Attack"] = min(s["chest_pain"], s["shortness_of_breath"], s["sweating"])
        raw_scores["Hypertension"] = min(mf["bp_hypertension"], mf["headache_high"])

        raw_scores["Diabetes"] = min(mf["thirst_high"], s["urination_freq"], mf["fatigue_high"])
        raw_scores["Thyroid Disorder"] = min(s["weight_loss"], s["appetite_loss"], mf["fatigue_high"])

        raw_scores["Allergic Reaction"] = min(max(s["skin_rash"], s["rash"]), s["itching"])
        raw_scores["Conjunctivitis"] = min(s["eye_redness"], s["watery_eyes"])
        raw_scores["Muscle Strain"] = min(mf["back_high"], mf["fatigue_high"])
        raw_scores["Arthritis"] = min(mf["joint_high"], s["stiffness"])

        low_symptom_level = 1.0 - max(
            mf["fever_high"], mf["cough_severe"], mf["fatigue_high"],
            mf["headache_high"], mf["nausea_high"], mf["stomach_high"],
            max(s.values())
        )
        raw_scores["Healthy"] = max(0.0, min(mf["fever_normal"], low_symptom_level))

        # apply differential contradiction penalties and specificity weighting
        rule_map = {r["disease"]: r for r in self.rules}
        scored_candidates = []

        for disease, base_val in raw_scores.items():
            if base_val <= 0.01:
                scored_candidates.append({
                    "disease": disease, "final_score": 0.0, "base_score": 0.0,
                    "spec": 1, "contra": []
                })
                continue

            r_meta = rule_map.get(disease, {})
            contra = check_contradictions(r_meta, p) if r_meta else []
            contra_penalty = min(0.70, len(contra) * 0.25)
            spec_count = len(r_meta.get("conditions", [])) if r_meta else 1
            final_val = base_val * (1.0 - contra_penalty)

            scored_candidates.append({
                "disease": disease,
                "final_score": round(float(final_val), 3),
                "base_score": round(float(base_val), 3),
                "spec": spec_count,
                "contra": contra
            })

        # tie-breaker: final_score, specificity, fewer contradictions, base_score
        scored_candidates.sort(
            key=lambda x: (x["final_score"], x["spec"], -len(x["contra"]), x["base_score"]),
            reverse=True
        )

        winner = scored_candidates[0]
        top_disease = winner["disease"]
        top_score = winner["final_score"]

        # defuzzification via center of gravity centroid
        sev_low = max(mf["fever_normal"], mf["spo2_normal"]) * 0.5
        sev_med = max(mf["fever_mild"], mf["cough_mod"], mf["fatigue_high"], mf["headache_high"])
        sev_high = max(mf["fever_extreme"], mf["spo2_hypoxic"], mf["hr_tachycardia"], mf["headache_severe"], mf["platelet_low"])

        num = sev_low * 15.0 + sev_med * 50.0 + sev_high * 90.0
        den = sev_low + sev_med + sev_high + 1e-6
        severity_index = round(float(num / den), 1)

        trace = [
            f"{c['disease']}: truth degree = {c['final_score']} (base: {c['base_score']}, contradictions: {len(c['contra'])})"
            for c in scored_candidates[:5] if c["final_score"] > 0.05
        ]
        if not trace:
            trace = ["Symptom membership below diagnostic activation threshold."]

        return {
            "method": "Fuzzy Logic",
            "diagnosis": top_disease if top_score >= 0.20 else "Healthy" if raw_scores["Healthy"] > 0.35 else "Undetermined / Inconclusive",
            "confidence": round(float(top_score), 2),
            "severity_score": severity_index,
            "ranked_diseases": [(c["disease"], c["final_score"]) for c in scored_candidates[:4]],
            "trace": trace,
            "reasoning": f"Fuzzy inference selected '{top_disease}' with truth degree {top_score} and severity index {severity_index}." + (f" Flagged contradictions: {', '.join(winner['contra'])}." if winner['contra'] else "")
        }

fuzzy_engine = FuzzyLogicEngine(CLINICAL_RULES)


## Theoretical Foundations of Diagnostic Approaches

### 1. Rule-Based Expert System
A rule-based expert system represents clinical domain knowledge as a formal set of production rules structured in an antecedent-consequent format (IF condition THEN action). In medical diagnosis, conditions describe patient observations such as body temperature thresholds, symptom presence, or lab values, while consequents state the inferred disease. When a patient profile is evaluated, an inference engine compares the patient findings against the antecedent of every rule in the knowledge base. Rules whose antecedent conditions evaluate to true are placed into a conflict set. When multiple rules match simultaneously, a conflict resolution strategy determines the winning deduction. In this implementation, specificity ordering resolves conflicts: rules requiring more specific clinical preconditions take precedence over broader screening rules.

The primary strength of rule-based systems lies in their transparency and deterministic execution. Every clinical recommendation directly maps to an explicit rule, allowing clinicians to inspect the exact logic behind any diagnosis. Unlike statistical or black-box machine learning models, production rules can be audited, edited, and verified by domain experts without retraining. However, rule-based architectures face limitations when dealing with borderline clinical measurements or noisy data. A strict threshold such as fever > 38.0 degrees Celsius will reject a patient presenting at 37.9 degrees Celsius, even if all other indicators align with the disease. Maintaining large rule sets also introduces engineering overhead, as conflicting or redundant rules can degrade inference stability over time.

### 2. Forward Chaining
Forward chaining is a data-driven inference technique that begins with an initial set of known facts in working memory and works forward toward a terminal conclusion. The inference engine operates through an iterative match-resolve-act cycle. During each cycle, the system matches known facts against rule premises. If a premise is satisfied, the rule fires and adds new derived assertions into working memory. These intermediate assertions can represent clinical abstractions, such as classifying a raw body temperature of 38.5 degrees Celsius as hyperthermia, or grouping symptoms into an acute respiratory syndrome. Subsequent cycles match both base facts and derived abstractions against higher-level diagnostic rules until no further rules can fire.

This bottom-up approach aligns naturally with real-world medical consultations where a clinician collects raw symptoms, lab panels, and vital signs before formulating a differential diagnosis. Forward chaining handles opportunistic discovery effectively because it evaluates all available data without presupposing a specific disease outcome. The execution trace reveals each intermediate deduction step, offering a verifiable reasoning chain from vital sign acquisition to terminal disease confirmation. A practical drawback of pure forward chaining is computational redundancy: the engine may fire rules and derive intermediate facts that are irrelevant to the final diagnosis. In expansive clinical ontologies, unrestrained forward inference requires agenda management and prioritization heuristics to avoid unbounded combinatorial expansion.

### 3. Backward Chaining
Backward chaining is a goal-driven inference technique that begins with a hypothesized disease conclusion and works backward to determine whether observed patient data supports it. The system selects a target hypothesis, such as Flu or Pneumonia, and queries the knowledge base for rules that conclude this disease. The engine then treats each precondition of the matching rule as an unresolved sub-goal. If a sub-goal corresponds to a primary clinical fact such as a recorded SpO2 value, the engine checks working memory. If a sub-goal depends on a deduced clinical state, the engine suspends the top-level goal and recursively attempts to prove the intermediate assertion using the same backward mechanism.

This top-down structure mirrors the clinical diagnostic process of hypothesis testing, where a physician suspects a particular condition and orders specific confirmatory tests. Backward chaining demonstrates high query efficiency because it only evaluates rules and facts directly relevant to proving or refuting the active goal, ignoring unassociated symptoms. It produces structured proof trees that clearly communicate why a hypothesis succeeded or which specific criterion caused it to fail. The limitation of backward chaining is that it requires an initial hypothesis to evaluate. If the candidate goal space is large and unranked, testing every potential condition sequentially can lead to exhaustive search overhead. In practice, backward chaining functions best when paired with an initial heuristic filter or clinician-selected target.

### 4. Fuzzy Logic
Fuzzy logic extends classical binary logic to accommodate the continuous, ambiguous nature of biological and clinical variables. Rather than forcing continuous vital signs into crisp boundaries where 37.9 degrees Celsius is categorized as normal and 38.0 degrees Celsius as febrile, fuzzy sets define degrees of truth between 0.0 and 1.0. A fuzzification layer maps crisp numeric inputs such as temperature, oxygen saturation, and symptom severity into linguistic variables like mild, moderate, or severe through triangular and trapezoidal membership functions. A Mamdani inference engine then evaluates fuzzy production rules using t-norm (minimum) operators for logical conjunction and s-norm (maximum) operators for disjunction, yielding a continuous membership distribution across candidate diagnoses.

The defuzzification step aggregates the qualified output surfaces and converts them into a single crisp severity index and confidence ranking, typically through the center of gravity centroid calculation. This architecture handles borderline clinical presentations and measurement noise gracefully, avoiding the brittle cliff effects common in boolean rule systems. A patient with borderline temperature and high cough intensity still receives a proportionate likelihood score rather than an immediate negative classification. The trade-off lies in calibration complexity. Defining appropriate membership function boundaries and tuning fuzzy rule weights requires extensive empirical validation and domain consensus. Without systematic calibration, overlapping membership functions can generate diffuse likelihood distributions that reduce clinical specificity.


In [ ]:
# benchmarking the four methods on the 200 patient dataset
results = []
ground_truths = []
equiv_map = {
    "COVID-19": ["COVID-19", "Pneumonia"],
    "Dengue Fever": ["Dengue Fever", "Dengue"]
}

n_patients = len(df)
latencies = {"Rule-Based": [], "Forward Chaining": [], "Backward Chaining": [], "Fuzzy Logic": []}
correct_counts = {"Rule-Based": 0, "Forward Chaining": 0, "Backward Chaining": 0, "Fuzzy Logic": 0}
coverage_counts = {"Rule-Based": 0, "Forward Chaining": 0, "Backward Chaining": 0, "Fuzzy Logic": 0}

for idx, row in df.iterrows():
    p_dict = row.to_dict()
    gt = str(row.get("diagnosis", "Unknown")).strip()
    ground_truths.append(gt)
    valid_targets = equiv_map.get(gt, [gt])

    # benchmark rule-based
    t0 = time.perf_counter()
    res_rb = rbs_engine.diagnose(p_dict)
    lat_rb = (time.perf_counter() - t0) * 1e6
    latencies["Rule-Based"].append(lat_rb)
    if res_rb["diagnosis"] in valid_targets:
        correct_counts["Rule-Based"] += 1
    if res_rb["diagnosis"] != "Undetermined / Inconclusive":
        coverage_counts["Rule-Based"] += 1

    # benchmark forward chaining
    t0 = time.perf_counter()
    res_fc = fc_engine.infer(p_dict)
    lat_fc = (time.perf_counter() - t0) * 1e6
    latencies["Forward Chaining"].append(lat_fc)
    if res_fc["diagnosis"] in valid_targets:
        correct_counts["Forward Chaining"] += 1
    if res_fc["diagnosis"] != "Undetermined / Inconclusive":
        coverage_counts["Forward Chaining"] += 1

    # benchmark backward chaining
    t0 = time.perf_counter()
    res_bc = bc_engine.infer(p_dict)
    lat_bc = (time.perf_counter() - t0) * 1e6
    latencies["Backward Chaining"].append(lat_bc)
    if res_bc["diagnosis"] in valid_targets:
        correct_counts["Backward Chaining"] += 1
    if res_bc["diagnosis"] != "Undetermined / Inconclusive":
        coverage_counts["Backward Chaining"] += 1

    # benchmark fuzzy logic
    t0 = time.perf_counter()
    res_fz = fuzzy_engine.infer(p_dict)
    lat_fz = (time.perf_counter() - t0) * 1e6
    latencies["Fuzzy Logic"].append(lat_fz)
    if res_fz["diagnosis"] in valid_targets:
        correct_counts["Fuzzy Logic"] += 1
    if res_fz["diagnosis"] != "Undetermined / Inconclusive":
        coverage_counts["Fuzzy Logic"] += 1

# generate benchmark summary table
summary_data = []
for m in ["Rule-Based", "Forward Chaining", "Backward Chaining", "Fuzzy Logic"]:
    acc = (correct_counts[m] / n_patients) * 100.0 if n_patients else 0.0
    cov = (coverage_counts[m] / n_patients) * 100.0 if n_patients else 0.0
    avg_lat = np.mean(latencies[m])
    summary_data.append({
        "Method": m,
        "Accuracy (%)": round(acc, 2),
        "Diagnostic Coverage (%)": round(cov, 2),
        "Mean Latency (microseconds)": round(avg_lat, 2)
    })

benchmark_df = pd.DataFrame(summary_data)
print("Diagnostic Method Comparison Matrix (Evaluated on 200 Patient Records):")
print(benchmark_df.to_string(index=False))

# visualize benchmark results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

methods = benchmark_df["Method"]
colors = ["#2b5c8f", "#2a9d8f", "#e76f51", "#f4a261"]

ax1.bar(methods, benchmark_df["Accuracy (%)"], color=colors, width=0.55)
ax1.set_title("Diagnostic Match Rate (%)")
ax1.set_ylabel("Accuracy (%)")
ax1.set_ylim(0, 105)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

ax2.bar(methods, benchmark_df["Mean Latency (microseconds)"], color=colors, width=0.55)
ax2.set_title("Inference Latency per Patient (microseconds)")
ax2.set_ylabel("Latency (microseconds)")
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## Architectural Comparison Matrix

| Evaluation Dimension | Rule-Based Expert System | Forward Chaining | Backward Chaining | Fuzzy Logic (Mamdani) |
| :--- | :--- | :--- | :--- | :--- |
| Reasoning Direction | Direct pattern matching | Data-driven (bottom-up) | Goal-driven (top-down) | Approximate / Soft reasoning |
| Primary Input Handling | Crisp boolean comparisons | Asserted working memory facts | Target hypothesis queries | Fuzzified membership degrees [0, 1] |
| Conflict Resolution | Precondition specificity | Agenda firing priority | Goal tree ordering | Centroid defuzzification |
| Uncertainty Management | Deterministic (all-or-nothing) | Deterministic (fact chaining) | Deterministic (subgoal proof) | Continuous truth degrees |
| Explainability Output | Fired rule identifier and text | Sequential cycle derivation trace | Hierarchical subgoal proof tree | Ranked membership scores and centroid |
| Missing Input Tolerance | Low (unmatched antecedents fail) | Moderate (fires partial abstractions) | High (localizes missing clinical tests) | High (evaluates partial fuzzy support) |
| Execution Complexity | O(R) linear rule scan | O(R * C) multi-cycle propagation | O(R * G) goal tree evaluation | O(R * F) membership calculation |


In [ ]:
import gradio as gr

# clinical presets covering diverse disease presentations
PRESET_CASES = {
    "Patient: Flu (Acute)": {
        "fever": 38.6, "cough": 8, "spo2": 96, "hr": 84, "bp": 120, "platelet": 250000,
        "fatigue": 5, "headache": 4, "nausea": 1, "stomach_pain": 0, "joint_pain": 2, "back_pain": 1, "thirst": 2,
        "urination": "normal",
        "rash": False, "bleeding": False, "diarrhea": False, "vomiting": False,
        "sore_throat": False, "body_pain": False, "wheezing": False, "chest_pain": False,
        "shortness_of_breath": False, "night_sweats": False, "stiff_neck": False,
        "dizziness": False, "confusion": False, "sweating": False, "itching": False,
        "eye_redness": False, "watery_eyes": False, "stiffness": False,
        "appetite_loss": False, "weight_loss": False
    },
    "Patient: Pneumonia (Respiratory Distress)": {
        "fever": 38.8, "cough": 9, "spo2": 89, "hr": 98, "bp": 124, "platelet": 240000,
        "fatigue": 8, "headache": 4, "nausea": 1, "stomach_pain": 0, "joint_pain": 1, "back_pain": 1, "thirst": 3,
        "urination": "normal",
        "rash": False, "bleeding": False, "diarrhea": False, "vomiting": False,
        "sore_throat": False, "body_pain": False, "wheezing": False, "chest_pain": False,
        "shortness_of_breath": True, "night_sweats": False, "stiff_neck": False,
        "dizziness": False, "confusion": False, "sweating": False, "itching": False,
        "eye_redness": False, "watery_eyes": False, "stiffness": False,
        "appetite_loss": False, "weight_loss": False
    },
    "Patient: Meningitis (Neurological Crisis)": {
        "fever": 38.7, "cough": 2, "spo2": 97, "hr": 92, "bp": 130, "platelet": 230000,
        "fatigue": 7, "headache": 9, "nausea": 4, "stomach_pain": 0, "joint_pain": 2, "back_pain": 2, "thirst": 3,
        "urination": "normal",
        "rash": False, "bleeding": False, "diarrhea": False, "vomiting": False,
        "sore_throat": False, "body_pain": False, "wheezing": False, "chest_pain": False,
        "shortness_of_breath": False, "night_sweats": False, "stiff_neck": True,
        "dizziness": True, "confusion": False, "sweating": False, "itching": False,
        "eye_redness": False, "watery_eyes": False, "stiffness": False,
        "appetite_loss": False, "weight_loss": False
    },
    "Patient: Food Poisoning (Gastrointestinal)": {
        "fever": 37.2, "cough": 1, "spo2": 98, "hr": 82, "bp": 118, "platelet": 250000,
        "fatigue": 6, "headache": 3, "nausea": 8, "stomach_pain": 7, "joint_pain": 0, "back_pain": 0, "thirst": 5,
        "urination": "normal",
        "rash": False, "bleeding": False, "diarrhea": True, "vomiting": True,
        "sore_throat": False, "body_pain": False, "wheezing": False, "chest_pain": False,
        "shortness_of_breath": False, "night_sweats": False, "stiff_neck": False,
        "dizziness": False, "confusion": False, "sweating": False, "itching": False,
        "eye_redness": False, "watery_eyes": False, "stiffness": False,
        "appetite_loss": False, "weight_loss": False
    },
    "Patient: Dengue Hemorrhagic Fever (Vascular)": {
        "fever": 39.4, "cough": 1, "spo2": 96, "hr": 110, "bp": 105, "platelet": 45000,
        "fatigue": 5, "headache": 7, "nausea": 3, "stomach_pain": 4, "joint_pain": 5, "back_pain": 3, "thirst": 4,
        "urination": "normal",
        "rash": True, "bleeding": True, "diarrhea": False, "vomiting": False,
        "sore_throat": False, "body_pain": False, "wheezing": False, "chest_pain": False,
        "shortness_of_breath": False, "night_sweats": False, "stiff_neck": False,
        "dizziness": True, "confusion": False, "sweating": False, "itching": False,
        "eye_redness": False, "watery_eyes": False, "stiffness": False,
        "appetite_loss": False, "weight_loss": False
    },
    "Patient: Healthy (Low Symptom Profile)": {
        "fever": 36.6, "cough": 1, "spo2": 99, "hr": 70, "bp": 115, "platelet": 260000,
        "fatigue": 1, "headache": 1, "nausea": 0, "stomach_pain": 0, "joint_pain": 0, "back_pain": 0, "thirst": 2,
        "urination": "normal",
        "rash": False, "bleeding": False, "diarrhea": False, "vomiting": False,
        "sore_throat": False, "body_pain": False, "wheezing": False, "chest_pain": False,
        "shortness_of_breath": False, "night_sweats": False, "stiff_neck": False,
        "dizziness": False, "confusion": False, "sweating": False, "itching": False,
        "eye_redness": False, "watery_eyes": False, "stiffness": False,
        "appetite_loss": False, "weight_loss": False
    }
}

def load_preset(name):
    if not name or name not in PRESET_CASES:
        return [
            37.0, 0, 98, 75, 120, 250000,
            0, 0, 0, 0, 0, 0, 0, "normal",
            False, False, False, False, False, False, False, False,
            False, False, False, False, False, False, False, False,
            False, False, False, False
        ]
    c = PRESET_CASES[name]
    return [
        c["fever"], c["cough"], c["spo2"], c["hr"], c["bp"], c["platelet"],
        c["fatigue"], c["headache"], c["nausea"], c["stomach_pain"], c["joint_pain"], c["back_pain"], c["thirst"], c["urination"],
        c["rash"], c["bleeding"], c["diarrhea"], c["vomiting"],
        c["sore_throat"], c["body_pain"], c["wheezing"], c["chest_pain"],
        c["shortness_of_breath"], c["night_sweats"], c["stiff_neck"],
        c["dizziness"], c["confusion"], c["sweating"], c["itching"],
        c["eye_redness"], c["watery_eyes"], c["stiffness"],
        c["appetite_loss"], c["weight_loss"]
    ]

def run_diagnostic_system(
    method, bc_target,
    fever, cough, spo2, hr, bp, platelet,
    fatigue, headache, nausea, stomach_pain, joint_pain, back_pain, thirst, urination,
    rash, bleeding, diarrhea, vomiting,
    sore_throat, body_pain, wheezing, chest_pain,
    shortness_of_breath, night_sweats, stiff_neck,
    dizziness, confusion, sweating, itching,
    eye_redness, watery_eyes, stiffness,
    appetite_loss, weight_loss
):
    patient_dict = {
        "fever_C": fever, "cough": cough, "oxygen_spo2": spo2, "heart_rate": hr, "blood_pressure": bp, "platelet": platelet,
        "fatigue": fatigue, "headache": headache, "nausea": nausea, "stomach_pain": stomach_pain,
        "joint_pain": joint_pain, "back_pain": back_pain, "thirst": thirst, "urination": urination,
        "rash": "yes" if rash else "no",
        "bleeding": "yes" if bleeding else "no",
        "diarrhea": "yes" if diarrhea else "no",
        "vomiting": "yes" if vomiting else "no",
        "sore_throat": "yes" if sore_throat else "no",
        "body_pain": "yes" if body_pain else "no",
        "wheezing": "yes" if wheezing else "no",
        "chest_pain": "yes" if chest_pain else "no",
        "shortness_of_breath": "yes" if shortness_of_breath else "no",
        "night_sweats": "yes" if night_sweats else "no",
        "stiff_neck": "yes" if stiff_neck else "no",
        "dizziness": "yes" if dizziness else "no",
        "confusion": "yes" if confusion else "no",
        "sweating": "yes" if sweating else "no",
        "skin_rash": "yes" if rash else "no",
        "itching": "yes" if itching else "no",
        "eye_redness": "yes" if eye_redness else "no",
        "watery_eyes": "yes" if watery_eyes else "no",
        "stiffness": "yes" if stiffness else "no",
        "appetite_loss": "yes" if appetite_loss else "no",
        "weight_loss": "yes" if weight_loss else "no"
    }

    if method == "Rule-Based Expert System":
        res = rbs_engine.diagnose(patient_dict)
        diag = res["diagnosis"]
        conf = f"{int(res['confidence'] * 100)}% (Certainty Factor)"
        fired_desc = "\n".join(f"- {r}" for r in res["fired_rules"]) if res["fired_rules"] else "None"
        details = f"Primary Rule: {res['primary_rule']}\nReasoning: {res['reasoning']}\n\nAll Fired Rules:\n{fired_desc}"
        trace_str = "\n".join(f"[Fired] {r}" for r in res["fired_rules"]) if res["fired_rules"] else "No production rules matched."

    elif method == "Forward Chaining":
        res = fc_engine.infer(patient_dict)
        diag = res["diagnosis"]
        conf = f"{int(res['confidence'] * 100)}% (Deduction CF)"
        details = f"Deduction Cycles: {res['cycles']}\nReasoning: {res['reasoning']}"
        trace_str = "\n".join(res["trace"])

    elif method == "Backward Chaining":
        res = bc_engine.infer(patient_dict, target_hypothesis=bc_target)
        diag = res["diagnosis"]
        conf = f"{int(res['confidence'] * 100)}% (Goal Proof CF)"
        details = f"Goal Tested: {res['goal_tested']}\nStatus: {'PROVEN' if res['proven'] else 'REFUTED'}\nReasoning: {res['reasoning']}"
        trace_str = "\n".join(res["trace"])

    else:
        res = fuzzy_engine.infer(patient_dict)
        diag = res["diagnosis"]
        conf = f"Truth Degree: {res['confidence']}"
        cand_desc = "\n".join(f"- {d}: degree {s}" for d, s in res["ranked_diseases"])
        details = f"Calculated Severity Index: {res['severity_score']} / 100\nReasoning: {res['reasoning']}\n\nTop Candidates:\n{cand_desc}"
        trace_str = "\n".join(res["trace"])

    r_all_rb = rbs_engine.diagnose(patient_dict)["diagnosis"]
    r_all_fc = fc_engine.infer(patient_dict)["diagnosis"]
    r_all_bc = bc_engine.infer(patient_dict)["diagnosis"]
    r_all_fz = fuzzy_engine.infer(patient_dict)["diagnosis"]

    comp_summary = (
        f"Cross-Method Agreement for this Patient Profile:\n"
        f"1. Rule-Based Expert System : {r_all_rb}\n"
        f"2. Forward Chaining         : {r_all_fc}\n"
        f"3. Backward Chaining        : {r_all_bc}\n"
        f"4. Fuzzy Logic (Mamdani)    : {r_all_fz}"
    )

    return diag, conf, details, trace_str, comp_summary

disease_list = ["Auto (Evaluate All Candidates)"] + sorted(list(set(r["disease"] for r in CLINICAL_RULES)))

with gr.Blocks(title="Clinical Expert Diagnosis System") as demo:
    gr.Markdown("# Clinical Medical Diagnosis Expert System")
    gr.Markdown("Unified Diagnostic Decision Support System comparing Rule-Based, Forward Chaining, Backward Chaining, and Fuzzy Logic.")

    with gr.Row():
        with gr.Column(scale=1):
            method_dropdown = gr.Dropdown(
                choices=["Rule-Based Expert System", "Forward Chaining", "Backward Chaining", "Fuzzy Logic"],
                value="Rule-Based Expert System",
                label="Diagnostic Inference Method"
            )
            bc_target_dropdown = gr.Dropdown(
                choices=disease_list,
                value="Auto (Evaluate All Candidates)",
                label="Backward Chaining Target Hypothesis (Ignored if other method)"
            )
            preset_dropdown = gr.Dropdown(
                choices=list(PRESET_CASES.keys()),
                label="Load Preset Patient Case",
                value="Patient: Flu (Acute)"
            )

        with gr.Column(scale=2):
            with gr.Row():
                fever_input = gr.Slider(minimum=35.0, maximum=42.0, value=38.6, step=0.1, label="Body Temperature (°C)")
                cough_input = gr.Slider(minimum=0, maximum=10, value=8, step=1, label="Cough Intensity (0-10)")
                spo2_input = gr.Slider(minimum=70, maximum=100, value=96, step=1, label="Oxygen Saturation SpO2 (%)")
            with gr.Row():
                hr_input = gr.Slider(minimum=40, maximum=180, value=84, step=1, label="Heart Rate (bpm)")
                bp_input = gr.Slider(minimum=80, maximum=220, value=120, step=1, label="Blood Pressure Systolic (mmHg)")
                platelet_input = gr.Slider(minimum=20000, maximum=450000, value=250000, step=5000, label="Platelet Count (cells/mcL)")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Symptom Severity Ratings (0-10)")
            fatigue_input = gr.Slider(0, 10, value=7, step=1, label="Fatigue")
            headache_input = gr.Slider(0, 10, value=4, step=1, label="Headache")
            nausea_input = gr.Slider(0, 10, value=1, step=1, label="Nausea")
            stomach_input = gr.Slider(0, 10, value=0, step=1, label="Stomach Pain")
            joint_input = gr.Slider(0, 10, value=2, step=1, label="Joint Pain")
            back_input = gr.Slider(0, 10, value=1, step=1, label="Back Pain")
            thirst_input = gr.Slider(0, 10, value=2, step=1, label="Thirst Level")
            urination_input = gr.Dropdown(choices=["normal", "frequent"], value="normal", label="Urination Frequency")

        with gr.Column():
            gr.Markdown("### Clinical Signs & Physical Exam")
            with gr.Row():
                rash_cb = gr.Checkbox(label="Rash", value=False)
                bleeding_cb = gr.Checkbox(label="Bleeding", value=False)
                diarrhea_cb = gr.Checkbox(label="Diarrhea", value=False)
                vomiting_cb = gr.Checkbox(label="Vomiting", value=False)
            with gr.Row():
                sore_throat_cb = gr.Checkbox(label="Sore Throat", value=False)
                body_pain_cb = gr.Checkbox(label="Body Pain", value=True)
                wheezing_cb = gr.Checkbox(label="Wheezing", value=False)
                chest_pain_cb = gr.Checkbox(label="Chest Pain", value=False)
            with gr.Row():
                sob_cb = gr.Checkbox(label="Shortness of Breath", value=False)
                sweats_cb = gr.Checkbox(label="Night Sweats", value=False)
                stiff_neck_cb = gr.Checkbox(label="Stiff Neck", value=False)
                dizziness_cb = gr.Checkbox(label="Dizziness", value=False)
            with gr.Row():
                confusion_cb = gr.Checkbox(label="Confusion", value=False)
                sweating_cb = gr.Checkbox(label="Sweating", value=False)
                itching_cb = gr.Checkbox(label="Itching", value=False)
                stiffness_cb = gr.Checkbox(label="Joint Stiffness", value=False)
            with gr.Row():
                eye_redness_cb = gr.Checkbox(label="Eye Redness", value=False)
                watery_eyes_cb = gr.Checkbox(label="Watery Eyes", value=False)
                appetite_loss_cb = gr.Checkbox(label="Appetite Loss", value=False)
                weight_loss_cb = gr.Checkbox(label="Unexplained Weight Loss", value=False)

    diagnose_btn = gr.Button("Execute Clinical Diagnosis", variant="primary")

    gr.Markdown("## Diagnostic Findings")
    with gr.Row():
        diag_output = gr.Textbox(label="Primary Diagnosis", interactive=False)
        conf_output = gr.Textbox(label="Confidence / Degree", interactive=False)
    with gr.Row():
        details_output = gr.Textbox(label="Reasoning & Findings", lines=5, interactive=False)
        trace_output = gr.Textbox(label="Execution / Inference Trace", lines=5, interactive=False)
    comp_output = gr.Textbox(label="Comparative Engine Agreement", lines=4, interactive=False)

    inputs_list = [
        method_dropdown, bc_target_dropdown,
        fever_input, cough_input, spo2_input, hr_input, bp_input, platelet_input,
        fatigue_input, headache_input, nausea_input, stomach_input, joint_input, back_input, thirst_input, urination_input,
        rash_cb, bleeding_cb, diarrhea_cb, vomiting_cb,
        sore_throat_cb, body_pain_cb, wheezing_cb, chest_pain_cb,
        sob_cb, sweats_cb, stiff_neck_cb,
        dizziness_cb, confusion_cb, sweating_cb, itching_cb,
        eye_redness_cb, watery_eyes_cb, stiffness_cb,
        appetite_loss_cb, weight_loss_cb
    ]

    outputs_list = [diag_output, conf_output, details_output, trace_output, comp_output]

    diagnose_btn.click(fn=run_diagnostic_system, inputs=inputs_list, outputs=outputs_list)

    preset_inputs = [
        fever_input, cough_input, spo2_input, hr_input, bp_input, platelet_input,
        fatigue_input, headache_input, nausea_input, stomach_input, joint_input, back_input, thirst_input, urination_input,
        rash_cb, bleeding_cb, diarrhea_cb, vomiting_cb,
        sore_throat_cb, body_pain_cb, wheezing_cb, chest_pain_cb,
        sob_cb, sweats_cb, stiff_neck_cb,
        dizziness_cb, confusion_cb, sweating_cb, itching_cb,
        eye_redness_cb, watery_eyes_cb, stiffness_cb,
        appetite_loss_cb, weight_loss_cb
    ]
    preset_dropdown.change(fn=load_preset, inputs=[preset_dropdown], outputs=preset_inputs)

# launch interface
demo.launch(share=True)
